# Adaptive RAG 와 Human-in-the-Loop

## Adaptive RAG 란? 적용하면 무엇이 좋은가
**개념**
- **Adaptive RAG** 는 질문을 먼저 **분석(라우팅)** 해서, 질문에 맞는 검색 전략을 골라 실행하는 RAG 패턴입니다. 모든 질문에 같은 검색을 하는 기본 RAG 와 달리 "메뉴 DB / 와인 DB / 웹 검색 / 검색 없이 LLM 답변" 중 **알맞은 경로를 선택**합니다.
- 이 노트북에서는 LLM 의 구조화 출력(`with_structured_output`)으로 라우터를 만들고, 그 결과로 조건부 엣지가 다음 노드를 정합니다.

**적용하면 얻는 효과**
- **검색 정확도 향상**: 메뉴 질문은 메뉴 DB, 와인 질문은 와인 DB, 최신 정보는 웹에서 찾아 **엉뚱한 문서가 섞이는 것을 줄입니다.**
- **불필요한 검색 제거**: 인사·일반 상식처럼 검색이 필요 없는 질문은 바로 답해 **비용과 지연을 줄입니다.**
- **환각 감소**: 답변을 검색된 문서에 근거해 생성하므로 근거 없는 내용을 지어낼 가능성이 낮아집니다.
- **예측 가능하고 디버깅이 쉬움**: 경로가 그래프에 명시되어 어느 경로를 탔는지 추적할 수 있습니다. (ReAct 처럼 LLM 이 자율적으로 도구를 부르는 방식보다 유연성은 낮음)
- 한계: **라우터의 분류 정확도에 의존**하며, 잘못 분류되면 엉뚱한 검색 경로로 갑니다. 라우팅용 LLM 호출이 한 번 추가됩니다.

## Human-in-the-Loop 란? 적용하면 무엇이 좋은가
**개념**
- **Human-in-the-Loop(HITL)** 는 자동화된 흐름 중간에 **사람이 개입**해 확인·수정·승인한 뒤 이어서 실행하는 방식입니다.
- LangGraph 에서는 `interrupt_before` 로 특정 노드 **직전에 실행을 멈추고**, 체크포인터에 저장된 상태를 `get_state` 로 확인, `update_state` 로 수정한 다음 **같은 `thread_id` 로 재개**합니다. (중단·재개에는 체크포인터가 필요합니다.)

**적용하면 얻는 효과**
- **품질·안전 확보**: 답변 생성 전에 검색 결과가 적절한지 사람이 확인하고, 잘못된 문서는 고쳐서 넘길 수 있습니다.
- **오류 전파 차단**: 잘못된 검색 결과가 답변으로 굳어지기 전에 중간에서 바로잡습니다.
- **통제권 유지**: 결제·삭제·외부 발송처럼 **되돌리기 어려운 작업은 승인 후에만** 실행하도록 할 수 있습니다.
- **재실행 비용 절감**: 처음부터 다시 돌리지 않고 중단 지점부터 이어서 실행합니다.
- 한계: 사람의 응답을 기다리므로 **완전 자동화보다 느리고**, 중단된 상태를 보관할 체크포인터(운영에서는 영구 저장소)가 필요합니다.

## 이 노트북의 역할
질문을 **분석해 알맞은 검색 방법을 고르는(Adaptive RAG)** 그래프를 만들고, 답변 생성 직전에 **사람이 개입(Human-in-the-Loop, HITL)** 해 검색 결과를 확인·수정한 뒤 이어서 실행하는 방법을 익힙니다.

| 단계 | 내용 | 배우는 것 |
|---|---|---|
| 1. 환경 설정 | API 키, `get_llm()`, 라이브러리 | 제공자 선택 |
| 2. Tool 정의 | 메뉴·와인(FAISS), 웹(Tavily) 검색 도구 | 검색 도구 3종, LLM 의 도구 선택 확인 |
| 3-1. Adaptive RAG | 구조화 출력 라우터 → 검색 → 답변 생성 | `with_structured_output`, 시작점 조건부 엣지 |
| 3-2. HITL | 중단점(`interrupt_before`), 상태 확인·수정, 재개 | 체크포인터, `get_state`, `update_state` |

## 전체 흐름 (Adaptive RAG 그래프)
```text
START ─(라우터: 질문 분석)─┬─ search_menu ─┐
                          ├─ search_wine ─┼→ generate → END
                          ├─ search_web  ─┘
                          └─ llm_fallback ─────────→ END   (검색이 필요 없는 질문)
```

| 노드 | 하는 일 | 상태 변경 |
|---|---|---|
| (라우터) | LLM 이 질문을 보고 `search_menu` / `search_wine` / `search_web` / `llm_fallback` 중 하나를 선택 | (노드가 아닌 **엣지의 조건**) |
| `search_*` | 선택된 도구로 검색 | `documents` |
| `generate` | 검색 문서에 근거해 답변 생성 | `generation` |
| `llm_fallback` | 문서 없이 LLM 일반 지식으로 답변 | `generation` |

> 이전 노트북(`04_ReAct_Memory`)의 에이전트는 **LLM 이 도구를 자율 호출**하지만, 이 노트북은 **라우터가 경로를 명시적으로 고르는** 워크플로우입니다. 경로가 예측 가능하고 디버깅이 쉬운 대신 유연성은 낮습니다.

## 사용 모델 (`get_llm()`)
- `PROVIDER` 값(`"groq"` / `"openai"` / `"upstage"`, 기본값 `"upstage"`)만 바꾸면 이후 모든 셀의 LLM 이 바뀝니다. 바꾼 뒤에는 LLM 설정 셀부터 다시 실행하세요.
- **임베딩은 항상 Upstage** (`solar-embedding-1-large`) 입니다. `../db/menu_db`, `../db/wine_db` 를 이 모델로 만들기 때문에 다른 임베딩으로 바꾸면 검색이 되지 않습니다. 따라서 어떤 제공자를 골라도 `UPSTAGE_API_KEY` 는 필요합니다.
- 웹 검색 도구에는 `TAVILY_API_KEY` 가 필요합니다.
- **사전 준비**: `../db/menu_db` 는 `01_LangGraph_StateGraph` 노트북이 만들어 저장합니다(없으면 01 을 먼저 실행). `../db/wine_db` 는 이 노트북의 2단계에서 `../data/restaurant_wine.txt` 로 만듭니다(이미 있으면 건너뜀).
- LLM·임베딩·웹 검색 호출은 사용 요금이 발생할 수 있습니다.

## 1. 환경 설정

`(1) API 키`

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 제공자별 API 키 (키 값은 출력하지 않음)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")  # 임베딩에 항상 필요

# Tavily 는 환경변수 TAVILY_API_KEY 를 라이브러리가 직접 읽음 — 설정 여부만 확인
print("TAVILY_API_KEY 설정됨:", bool(os.getenv("TAVILY_API_KEY")))

`(3) LLM 설정 (Groq / OpenAI / Upstage 선택)`
- `PROVIDER` 값만 바꾸면 이후 모든 셀의 모델이 바뀝니다. (Groq 는 OpenAI 호환 API 라서 `ChatOpenAI` 에 `base_url` 만 다르게 연결)
- 바꾼 뒤에는 이 셀부터 다시 실행하세요. OpenAI 선택 시 사용 요금이 발생합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage

# 사용할 LLM 제공자 선택: "groq", "openai", "upstage" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "upstage"

# 제공자별 설정 (cls: 사용할 LangChain 채팅 모델 클래스)
LLM_CONFIGS = {
    "groq": {
        "cls": ChatOpenAI,  # Groq 는 OpenAI 호환 API 라서 base_url 만 다름
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "cls": ChatOpenAI,
        "api_key": OPENAI_API_KEY,
        "base_url": None,  # 생략 시 OpenAI 기본 엔드포인트 사용
        "model": "gpt-4o-mini",  # 테스트에는 작은 모델 사용 (대안: "gpt-4o")
    },
    "upstage": {
        "cls": ChatUpstage,
        "api_key": UPSTAGE_API_KEY,
        "base_url": "https://api.upstage.ai/v1",
        "model": "solar-pro3",
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs):
    """선택한 제공자(기본값: PROVIDER)의 채팅 모델 인스턴스를 생성합니다.
    verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: 모델 클래스에 그대로 전달할 추가 옵션 (예: seed=42, timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = dict(LLM_CONFIGS[provider])
    model_cls = config.pop("cls")
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = model_cls(**config, temperature=temperature, max_retries=MAX_RETRIES, **kwargs)
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
# 라우팅·구조화 출력은 결과가 일정해야 하므로 temperature=0 사용
llm = get_llm(temperature=0)

`(2) 기본 라이브러리`

In [ ]:
import warnings
from pprint import pprint
from textwrap import dedent
from typing import List, Literal, TypedDict

warnings.filterwarnings("ignore")

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langchain_upstage import UpstageEmbeddings
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

## 2. Tool 정의

In [ ]:
# 상수: 벡터DB 가 Upstage 임베딩으로 만들어져 있으므로 같은 모델을 사용해야 함
EMBEDDING_MODEL = "solar-embedding-1-large"
MENU_DB_PATH = "../db/menu_db"
WINE_DB_PATH = "../db/wine_db"
WINE_DATA_PATH = "../data/restaurant_wine.txt"  # 와인 리스트 원본
REBUILD_WINE_DB = False  # True 이면 이미 있는 와인 저장소도 다시 만듦 (임베딩 요금 발생)
VECTOR_SEARCH_K = 6        # 벡터DB 검색 시 가져올 문서 수
WEB_SEARCH_MAX_RESULTS = 2  # 웹 검색 결과 수

embeddings_model = UpstageEmbeddings(model=EMBEDDING_MODEL)

`(1) 와인 벡터저장소 준비`
* 메뉴 저장소(`menu_db`)는 `01_LangGraph_StateGraph` 에서 만들었습니다. 와인 저장소(`wine_db`)는 여기서 만듭니다.
* 와인 리스트도 메뉴판과 같은 `1. 이름 ...` 형식이라 **같은 방식으로 항목별로 분할**합니다. 이미 저장소가 있으면 건너뜁니다.

In [ ]:
import re
from typing import List

from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document


def split_menu_items(document: Document) -> List[Document]:
    """메뉴 텍스트 문서를 '1. 메뉴명 ...' 항목 단위로 나눠, 항목마다 Document 하나로 만듭니다.

    metadata: source(원본 파일 경로), menu_number(순번, 1부터), menu_name(메뉴 이름)
    """
    # '숫자. ' 로 시작해 다음 '\n\n숫자.' 또는 문서 끝 직전까지 (re.DOTALL: '.' 이 줄바꿈도 포함)
    pattern = r"(\d+\.\s.*?)(?=\n\n\d+\.|$)"
    menu_items = re.findall(pattern, document.page_content, re.DOTALL)

    menu_documents = []
    for i, item in enumerate(menu_items, 1):
        # 첫 줄 '1. 시그니처 스테이크' 에서 '숫자.' 를 떼고 메뉴 이름만 추출
        menu_name = item.split("\n")[0].split(".", 1)[1].strip()
        menu_documents.append(Document(
            page_content=item.strip(),
            metadata={"source": document.metadata["source"], "menu_number": i, "menu_name": menu_name},
        ))
    return menu_documents


if os.path.exists(WINE_DB_PATH) and not REBUILD_WINE_DB:
    print(f"[건너뜀] 이미 있음: {WINE_DB_PATH}")
else:
    # 와인 리스트를 항목별로 분할 후 임베딩해 저장 (임베딩 API 호출 → 사용 요금 발생)
    wine_documents = []
    for doc in TextLoader(WINE_DATA_PATH, encoding="utf-8").load():
        wine_documents += split_menu_items(doc)

    os.makedirs(os.path.dirname(WINE_DB_PATH), exist_ok=True)
    FAISS.from_documents(documents=wine_documents, embedding=embeddings_model).save_local(WINE_DB_PATH)
    print(f"[저장] {len(wine_documents)}개 항목 → {WINE_DB_PATH}")

`(2) 검색 도구 3종`
* 저장된 벡터저장소를 불러와 검색 도구를 만듭니다.

In [ ]:
# 직접 만든 로컬 파일이므로 역직렬화를 허용
menu_db = FAISS.load_local(MENU_DB_PATH, embeddings_model, allow_dangerous_deserialization=True)
wine_db = FAISS.load_local(WINE_DB_PATH, embeddings_model, allow_dangerous_deserialization=True)


@tool
def search_menu(query: str) -> List[Document]:
    """레스토랑 메뉴(음식 종류, 재료, 가격 등)에 대한 정보를 메뉴 데이터베이스에서 검색합니다.
    Use this tool only for restaurant menu-related queries."""
    docs = menu_db.similarity_search(query, k=VECTOR_SEARCH_K)
    return docs or [Document(page_content="관련 메뉴 정보를 찾을 수 없습니다.")]


@tool
def search_wine(query: str) -> List[Document]:
    """레스토랑 와인 리스트(품종, 산지, 페어링 등)에 대한 정보를 와인 데이터베이스에서 검색합니다.
    Use this tool only for restaurant wine-related queries."""
    docs = wine_db.similarity_search(query, k=VECTOR_SEARCH_K)
    return docs or [Document(page_content="관련 와인 정보를 찾을 수 없습니다.")]


@tool
def search_web(query: str) -> List[Document]:
    """데이터베이스에 없는 정보나 최신 정보를 인터넷에서 검색합니다.
    Searches the internet for information that does not exist in the database or for the latest information."""
    response = TavilySearch(max_results=WEB_SEARCH_MAX_RESULTS).invoke({"query": query})
    results = response.get("results", [])  # [{url, content, ...}]

    docs = [
        Document(
            page_content=f'<Document href="{item["url"]}"/>\n{item["content"]}\n</Document>',
            metadata={"source": "web search", "url": item["url"]},
        )
        for item in results
    ]
    return docs or [Document(page_content="관련 정보를 찾을 수 없습니다.")]


# 도구 목록
tool_list = [search_menu, search_wine, search_web]
print(type(tool_list[0]))

`(3) LLM 의 도구 선택 확인 (bind_tools)`
* `bind_tools` 로 도구를 연결하면 LLM 이 **스스로** 도구를 고릅니다. (04번 노트북의 에이전트 방식)
* 이 셀은 도구를 실행하지 않고 LLM 의 선택(`tool_calls`)만 확인합니다. 3-1 에서는 같은 일을 **라우터(구조화 출력)** 로 명시적으로 처리합니다.

In [ ]:
llm_with_tools = llm.bind_tools(tool_list)  # 도구 설명이 포함된 LLM

test_questions = [
    "대표 메뉴는 무엇인가요?",                       # 기대: search_menu
    "안녕하세요?",                                  # 기대: 도구 없음 (바로 답변)
    "2024년 상반기 엔비디아 시가총액은 어떻게 변동 되었나요?",  # 기대: search_web
]

for question in test_questions:
    ai_msg = llm_with_tools.invoke(question)
    print(f"질문: {question}")
    print("  tool_calls:", [(tc["name"], tc["args"]) for tc in ai_msg.tool_calls] or "없음")

## 3. Adaptive RAG

### 3-1. 그래프 구현

`(1) 상태 정의`

In [ ]:
class AdaptiveRagState(TypedDict):
    """Adaptive RAG 그래프의 상태"""
    question: str              # 사용자 질문
    documents: List[Document]  # 검색된 문서 (search_* 노드가 저장)
    generation: str            # 최종 답변 (generate / llm_fallback 노드가 저장)

`(2) 질문 분석 -> 라우팅`
* LLM 이 질문을 분석해 검색 방법을 선택합니다: **메뉴 검색 / 와인 검색 / 웹 검색 / 검색 없이 단순 답변**
* `with_structured_output(Pydantic 모델)`: LLM 출력이 정해진 구조(JSON 스키마)를 따르도록 강제합니다. `Literal` 로 허용 값을 제한하면 잘못된 이름이 나오지 않습니다.

In [ ]:
class ToolSelector(BaseModel):
    """사용자 질문을 가장 적절한 처리 경로로 라우팅하는 데이터 모델."""

    # Literal: 이 네 값 중 하나만 허용 (llm_fallback = 검색 없이 LLM 이 바로 답변)
    tool: Literal["search_menu", "search_wine", "search_web", "llm_fallback"] = Field(
        description="질문에 가장 적절한 경로: search_menu, search_wine, search_web, llm_fallback 중 하나",
    )


# 구조화 출력 LLM: 응답이 ToolSelector 객체로 반환됨
structured_llm = llm.with_structured_output(ToolSelector)

ROUTER_SYSTEM_PROMPT = dedent("""당신은 사용자 질문을 적절한 경로로 라우팅하는 데 특화된 AI 어시스턴트입니다.
다음 지침을 사용하십시오:
- 레스토랑의 메뉴에 대한 질문에는 search_menu 를 사용하십시오.
- 와인 추천이나 페어링 정보에는 search_wine 을 사용하십시오.
- 그 밖에 외부 정보나 최신 데이터가 필요한 질문에는 search_web 을 사용하십시오.
- 인사, 잡담처럼 검색이 필요 없는 질문에는 llm_fallback 을 사용하십시오.
항상 사용자 질문을 기반으로 가장 적절한 경로를 선택하십시오.""")

route_prompt = ChatPromptTemplate.from_messages([
    ("system", ROUTER_SYSTEM_PROMPT),
    ("human", "{question}"),  # 사용자 질문이 여기에 삽입됨
])

# 라우터 체인: 프롬프트 → 구조화 출력 LLM  (결과: ToolSelector)
question_router = route_prompt | structured_llm

# 라우팅 결과 확인 (기대값을 주석으로 표시)
router_tests = [
    "채식주의자를 위한 메뉴가 있나요?",              # search_menu
    "스테이크 메뉴와 어울리는 와인을 추천해주세요.",     # search_wine
    "2022년 월드컵 우승 국가는 어디인가요?",          # search_web
    "안녕하세요?",                                  # llm_fallback
]
for question in router_tests:
    print(f"{question} → {question_router.invoke({'question': question})}")

In [ ]:
def route_question_adaptive(state: AdaptiveRagState) -> Literal["search_menu", "search_wine", "search_web", "llm_fallback"]:
    """질문을 분석해 다음에 실행할 노드 이름을 반환하는 라우팅 함수 (조건부 엣지용).

    라우팅에 실패하면(예: 구조화 출력 오류) 안전하게 llm_fallback 으로 보냅니다.
    """
    try:
        return question_router.invoke({"question": state["question"]}).tool
    except Exception as e:
        print(f"Error in routing: {e}")
        return "llm_fallback"

`(3) 검색 노드`
* 세 검색 노드는 하는 일이 같고 **호출하는 도구만 다릅니다.** 그래서 노드를 만드는 함수(`make_search_node`)로 한 번에 만듭니다.
* 도구가 "결과 없음" 문서를 이미 반환하므로 노드에서는 별도 분기가 필요 없습니다.

In [ ]:
def make_search_node(search_tool):
    """주어진 검색 도구를 호출해 결과를 documents 로 저장하는 노드 함수를 만들어 반환합니다."""

    def search_node(state: AdaptiveRagState) -> dict:
        docs = search_tool.invoke(state["question"])
        return {"documents": docs}

    return search_node

search_menu_adaptive = make_search_node(search_menu)  # 레스토랑 메뉴 검색
search_wine_adaptive = make_search_node(search_wine)  # 와인 리스트 검색
search_web_adaptive = make_search_node(search_web)    # 웹 검색

`(4) 생성 노드`

In [ ]:
# RAG 프롬프트: 검색 문서만을 근거로 답변하도록 지시
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 제공된 문서를 기반으로 질문에 답변하는 Assistant입니다. 다음 지침을 따르십시오:

    1. 제공된 문서의 정보만을 사용하십시오.
    2. 문서에 관련 정보가 부족하면 "제공된 문서에는 이 질문에 답변할 정보가 포함되어 있지 않습니다."라고 말하십시오.
    3. 답변에 문서의 관련 부분을 인용하십시오.
    4. 추측하거나 문서에 없는 정보를 추가하지 마십시오.
    5. 답변을 간결하고 명확하게 유지하십시오.
    6. 관련 없는 정보는 생략하십시오."""),
    ("human", "다음 문서를 사용하여 다음 질문에 답하십시오:\n\n[문서]\n{documents}\n\n[질문]\n{question}"),
])


def generate_adaptive(state: AdaptiveRagState) -> dict:
    """검색된 문서를 근거로 최종 답변(generation)을 생성하는 노드."""
    question = state["question"]
    documents = state.get("documents", [])

    # 문서 목록을 프롬프트에 넣을 하나의 문자열로 변환 (본문 + 메타데이터)
    documents_text = "\n\n".join(
        f"---\n본문: {doc.page_content}\n메타데이터:{doc.metadata}\n---" for doc in documents
    )

    rag_chain = rag_prompt | llm | StrOutputParser()
    generation = rag_chain.invoke({"documents": documents_text, "question": question})
    return {"generation": generation}

In [ ]:
# LLM Fallback 프롬프트: 문서 없이 LLM 의 일반 지식으로 답변 ({documents} 변수가 없음)
fallback_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 다양한 주제를 돕는 AI 어시스턴트입니다. 다음 지침을 따르십시오:

    1. 능력이 닿는 한 정확하고 유용한 정보를 제공하십시오.
    2. 확신이 없을 때는 불확실성을 표현하십시오. 추측은 피하십시오.
    3. 답변은 간결하지만 정보를 담고 있도록 유지하십시오.
    4. 사용자에게 필요하면 명확한 설명을 요청할 수 있다고 알리십시오.
    5. 윤리적이고 건설적으로 응답하십시오.
    6. 해당되는 경우 신뢰할 수 있는 일반 출처를 언급하십시오."""),
    ("human", "{question}"),
])


def llm_fallback_adaptive(state: AdaptiveRagState) -> dict:
    """검색 없이 LLM 의 일반 지식으로 답변(generation)을 생성하는 노드."""
    llm_chain = fallback_prompt | llm | StrOutputParser()
    return {"generation": llm_chain.invoke({"question": state.get("question", "")})}

`(5) 그래프 연결`
* **시작점 `START` 에서 조건부 엣지**를 걸어 라우터가 첫 노드를 고르게 합니다.
* 매핑 `{반환값: 이동할 노드}` 를 적어 두면 mermaid 에 네 갈래가 정확히 그려집니다.

In [ ]:
builder = StateGraph(AdaptiveRagState)

# 1. 노드 추가
builder.add_node("search_menu", search_menu_adaptive)
builder.add_node("search_wine", search_wine_adaptive)
builder.add_node("search_web", search_web_adaptive)
builder.add_node("generate", generate_adaptive)
builder.add_node("llm_fallback", llm_fallback_adaptive)

# 2. 조건부 엣지: START → 라우팅 함수가 반환한 이름의 노드
builder.add_conditional_edges(
    START,
    route_question_adaptive,
    {
        "search_menu": "search_menu",
        "search_wine": "search_wine",
        "search_web": "search_web",
        "llm_fallback": "llm_fallback",
    },
)

# 3. 일반 엣지: 검색이 끝나면 항상 generate, 두 종류의 답변 생성 후 종료
builder.add_edge("search_menu", "generate")
builder.add_edge("search_wine", "generate")
builder.add_edge("search_web", "generate")
builder.add_edge("generate", END)
builder.add_edge("llm_fallback", END)

adaptive_rag = builder.compile()

In [ ]:
# 그래프 구조 확인: mermaid 코드 출력 (https://mermaid.live/ 에 붙여 넣으면 그림으로 보임)
mermaid_code = adaptive_rag.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

* https://mermaid.live/ 에서 `mermaid_code` 로 직접 확인할 수 있습니다. 점선(`-.->`)이 조건부 엣지입니다.
* 라우팅 함수 이름은 그림에 나타나지 않습니다. 분기는 노드가 아니라 엣지의 조건이기 때문입니다.

In [ ]:
# 이미지로 직접 보려면: 
from IPython.display import Image, display

display(Image(adaptive_rag.get_graph().draw_mermaid_png()))


In [ ]:
def doc_labels(docs: List[Document]) -> list:
    """문서 목록을 짧은 라벨(메뉴명·와인명·URL·출처) 목록으로 요약합니다. (출력 가독성용)"""
    keys = ("menu_name", "wine_name", "url", "source")
    return [next((d.metadata[k] for k in keys if d.metadata.get(k)), "(출처 없음)") for d in docs]


def run_stream(graph, inputs, config=None) -> None:
    """그래프를 stream 으로 실행하며 노드별 결과를 요약 출력합니다.
    inputs=None 이면 중단된 실행을 이어서 진행합니다."""
    for event in graph.stream(inputs, config=config):
        for node, update in event.items():
            if node.startswith("__"):  # __end__, __interrupt__ 등 내부 이벤트는 건너뜀
                continue
            print(f"[{node}]")
            if "documents" in update:
                print("  documents:", len(update["documents"]), "건 →", doc_labels(update["documents"]))
            if "generation" in update:
                print("  generation:", update["generation"])


run_stream(adaptive_rag, {"question": "스테이크 메뉴의 가격은 얼마인가요?"})  # 기대 경로: search_menu → generate

In [ ]:
run_stream(adaptive_rag, {"question": "푸이 퓌세 2019의 주요 품종은 무엇인가요?"})  # 기대 경로: search_wine → generate

In [ ]:
run_stream(adaptive_rag, {"question": "안녕하세요?"})  # 기대 경로: llm_fallback (검색 없음)

### 3-2. 사람의 개입 (Human-in-the-Loop)

* Human-in-the-Loop(HITL)는 AI 시스템에 사람의 판단과 개입을 통합하는 방식입니다. 자동화된 처리와 사람의 전문성을 결합해 결과의 정확성과 신뢰성을 높입니다.
* 여기서는 `generate` 노드 **직전에 멈춰서**, 사람이 검색 결과를 확인하고 필요하면 질문을 고친 뒤 이어서 실행합니다.

#### MemorySaver(체크포인터)의 역할
1. **상태 저장·복원**: 노드를 지날 때마다 그 시점의 상태를 저장하고, 같은 `thread_id` 로 다시 호출하면 마지막 상태에서 이어서 실행합니다. 중단점(breakpoint)이 가능한 이유입니다.
2. **메모리(RAM) 저장**: 빠르지만 Python 세션이 끝나면 사라집니다. 개발·테스트·데모용이며, 운영 환경에서는 DB 기반 체크포인터(예: `SqliteSaver`, `PostgresSaver`)를 사용합니다.
3. **대화 기록 유지**: 휘발성 메모리지만 같은 스레드의 다중 턴 대화를 이어갈 수 있습니다.

> 이 노트북은 정적 중단점(`interrupt_before`)을 사용합니다. 노드 **안에서** 사람에게 질문하고 응답을 받는 동적 방식은 `interrupt()` 함수(`Command(resume=...)`로 재개)를 사용합니다.

`(1) 체크포인트 설정`

In [ ]:
memory = MemorySaver()  # 인메모리 체크포인터

`(2) Breakpoint 추가`

In [ ]:
# 컴파일 시 체크포인터와 중단점을 지정: 'generate' 노드 실행 직전에 멈춤
adaptive_rag_hitl = builder.compile(
    checkpointer=memory,            # 모든 중간 상태를 memory 에 저장 (재개·검토에 필요)
    interrupt_before=["generate"],  # generate 실행 직전에 중단 → 검색 결과를 사람이 확인
)

print(type(adaptive_rag_hitl))
print(adaptive_rag_hitl.interrupt_before_nodes)

In [ ]:
mermaid_code = adaptive_rag_hitl.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

In [ ]:
# 이미지로 직접 보려면: 
from IPython.display import Image, display

display(Image(adaptive_rag_hitl.get_graph().draw_mermaid_png()))

`(3) Breakpoint 실행 확인`
* `thread_id` 를 지정해 실행하면 `generate` 직전에서 멈춥니다. 출력에는 `generate` 가 나타나지 않습니다.

In [ ]:
# 실행 스레드 지정: 같은 thread_id 로 호출해야 멈춘 지점의 상태를 이어받음
thread = {"configurable": {"thread_id": "breakpoint_test"}}

# 'search_menu' 까지 실행되고 generate 직전에서 멈춤
run_stream(adaptive_rag_hitl, {"question": "스테이크 메뉴의 가격은 얼마인가요?"}, thread)

`(4) Breakpoint 상태 확인`
* `get_state(thread)` 는 해당 스레드의 최신 체크포인트(`StateSnapshot`)를 돌려줍니다.
    * `values`: 현재 상태 / `next`: 다음에 실행될 노드(중단 상태에서는 `('generate',)`)

In [ ]:
current_state = adaptive_rag_hitl.get_state(thread)

print("다음에 실행될 노드:", current_state.next)
print("question  :", current_state.values["question"])
print("documents :", doc_labels(current_state.values["documents"]))
print("generation:", current_state.values.get("generation"))  # generate 전이므로 None

`(5) Breakpoint 이후 단계를 계속해서 실행`
* 입력을 `None` 으로 주면 새 입력 없이 **저장된 상태에서 이어서** 실행합니다.

In [ ]:
# None: 새로운 입력 없이 'breakpoint_test' 스레드의 멈춘 지점부터 재개
run_stream(adaptive_rag_hitl, None, thread)

In [ ]:
# 재개가 끝나면 다음 실행 노드가 없음 → next == ()
final_state = adaptive_rag_hitl.get_state(thread)
print("다음에 실행될 노드:", final_state.next)
print("최종 답변:")
print(final_state.values["generation"])

`(6) 상태 업데이트`
* 멈춘 동안 `update_state` 로 상태를 고칠 수 있습니다. 사람이 질문을 다듬어 다시 시도하게 하는 예입니다.
* **주의**: 질문만 고치면 `documents` 는 **이전 질문으로 검색한 결과 그대로**이고, 다음 노드(`generate`)만 새 질문으로 실행됩니다. 문서와 질문이 어긋나 "문서에 정보가 없다"는 답이 나오기 쉽습니다.
* 검색부터 다시 하려면 `as_node=START` 를 지정합니다. "START 노드가 이 값을 쓴 것"으로 기록되어, 다음 실행이 START 의 조건부 엣지(라우터)부터 다시 시작됩니다.

In [ ]:
# 새 스레드로 새 질문 실행 → generate 직전에서 멈춤
thread = {"configurable": {"thread_id": "breakpoint_update"}}
run_stream(adaptive_rag_hitl, {"question": "매운 음식이 있나요?"}, thread)

state = adaptive_rag_hitl.get_state(thread)
print("\n다음에 실행될 노드:", state.next)
print("question  :", state.values["question"])
print("generation:", state.values.get("generation"))

In [ ]:
# 상태 업데이트: 질문을 수정하고, as_node=START 로 라우팅·검색부터 다시 실행되도록 지정
adaptive_rag_hitl.update_state(thread, {"question": "매콤한 해산물 요리가 있나요?"}, as_node=START)

new_state = adaptive_rag_hitl.get_state(thread)
print("수정된 question:", new_state.values["question"])
print("다음에 실행될 노드:", new_state.next)  # 검색 노드부터 다시 실행됨

In [ ]:
# 입력을 None 으로 주면 저장된 상태부터 이어서 실행: 새 질문으로 검색 후 generate 직전에서 다시 멈춤
run_stream(adaptive_rag_hitl, None, thread)
print("\n다음에 실행될 노드:", adaptive_rag_hitl.get_state(thread).next)

In [ ]:
# 한 번 더 재개하면 generate 가 실행되어 최종 답변이 생성됨
run_stream(adaptive_rag_hitl, None, thread)

final_state = adaptive_rag_hitl.get_state(thread)
print("\n최종 답변:")
print(final_state.values["generation"])